# Notebook collaborative filtering

On regarde les préférences des utilisateurs et on recommande les articles les plus aimés par les utilisateurs similaires.

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import implicit

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

/Users/j/Documents/OC_Inge_IA/Projet_10/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#clicks = pd.read_csv("../news-portal-user-interactions-by-globocom/clicks_sample.csv")
clicks = pd.read_csv("../click_hour_concatenate_100.csv")
clicks.sample(20)

,user_id,session_id,session_start,session_size,click_article_id,click_timestamp,click_environment,click_deviceGroup,click_os,click_country,click_region,click_referrer_type
89195,30337,1506899941312974,1506899941000,6,48731,1506903302724,4,3,19,1,6,1
791232,154426,1507135013347722,1507135013000,9,97114,1507137332990,4,1,17,1,9,1
99977,33674,1506903713334202,1506903713000,2,161178,1506903731034,4,3,20,11,28,2
513848,123986,1507036404208959,1507036404000,6,255530,1507037862921,4,1,17,1,21,1
459193,78788,1507021658407358,1507021658000,2,160417,1507021763093,4,3,20,6,28,2
527744,112939,1507039907342590,1507039907000,2,186893,1507040054683,4,1,17,1,17,1
795022,76347,1507135747304040,1507135747000,8,361210,1507136654245,4,1,17,1,27,1
907288,8457,1507176390418643,1507176390000,3,286321,1507176453384,4,1,17,1,24,2
120338,39819,1506913164359057,1506913164000,5,313504,1506913388996,4,3,2,1,25,2
597701,138659,1507053810202476,1507053810000,2,166534,1507054041568,4,1,17,1,25,2


In [3]:
# 1. Prepare your data
# Assuming 'clicks' is your original dataframe. We create a 'rating' column (e.g., 1 for a click)
# If a user clicked the same article multiple times and you want to count that as the rating, you can do:
# df_surprise = clicks.groupby(['user_id', 'click_article_id']).size().reset_index(name='rating')
df_surprise = clicks[['user_id', 'click_article_id']].copy()
df_surprise['rating'] = 1 # We assign a rating of 1 for a click

# 2. Define a Reader
# Since our "ratings" are just 1s, the scale is [1, 1] (or [0, 1] if you include unclicked items)
reader = Reader(rating_scale=(1, 1))

# 3. Load the dataset into surprise
data = Dataset.load_from_df(df_surprise[['user_id', 'click_article_id', 'rating']], reader)

# 4. Split into train and test sets
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# 5. Initialize and train the algorithm (SVD is a very popular Matrix Factorization algorithm)
algo = SVD()
algo.fit(trainset)

# 6. Evaluate the model (Optional but recommended)
predictions = algo.test(testset)
print("RMSE:", accuracy.rmse(predictions))


RMSE: 0.0000
RMSE: 0.0


In [4]:
# Predict the score for a specific user and a specific article
user_id_to_predict = 0 # Replace with a real user_id from your dataset
article_id_to_predict = 12345 # Replace with a real click_article_id

prediction = algo.predict(uid=user_id_to_predict, iid=article_id_to_predict)
print(f"Predicted score for user {user_id_to_predict} on article {article_id_to_predict}: {prediction.est}")


Predicted score for user 0 on article 12345: 1


In [5]:
def get_top_n_recommendations(algo, user_id, df_original, n=5):
    # 1. Get all unique articles in the dataset
    all_articles = df_original['click_article_id'].unique()
    
    # 2. Get articles the user has ALREADY interacted with
    user_articles = df_original[df_original['user_id'] == user_id]['click_article_id'].unique()
    
    # 3. Find articles the user HAS NOT interacted with
    articles_to_predict = [art for art in all_articles if art not in user_articles]
    
    # 4. Predict the score for all unseen articles
    predictions = [algo.predict(user_id, article_id) for article_id in articles_to_predict]
    
    # 5. Sort predictions by the estimated score (highest first)
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # 6. Return the top N article IDs
    top_n_articles = [pred.iid for pred in predictions[:n]]
    return top_n_articles

# Example usage: Get top 5 recommendations for a specific user
recommended_articles = get_top_n_recommendations(algo, user_id=0, df_original=df_surprise, n=5)
print("Recommended Articles:", recommended_articles)


Recommended Articles: [np.int64(235840), np.int64(96663), np.int64(119592), np.int64(30970), np.int64(236065)]


In [6]:
# 1. Création du pivot
clicks_pivot = clicks.pivot_table(
    index='user_id', 
    columns='click_article_id', 
    aggfunc='size', 
    fill_value=0
)

# 2. Nettoyage : Suppression du nom de l'axe des colonnes
clicks_pivot.columns.name = None

# 3. (Optionnel mais recommandé) Remettre 'user_id' comme une colonne classique
clicks_pivot = clicks_pivot.reset_index()

# Affichage du résultat
display(clicks_pivot.head())


/var/folders/dm/pvk529ls4pg3bshgj9btlsc80000gn/T/ipykernel_14467/2435449585.py:2: PerformanceWarning: The following operation may generate 3023256640 cells in the resulting pandas object.
  clicks_pivot = clicks.pivot_table(


,user_id,27,81,137,143,271,290,343,388,442,...,363957,363967,363976,363980,363984,363992,364001,364017,364022,364046
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
clicks_pivot.shape

(176840, 17097)

In [8]:
# On peut rajouter une colonne temporaire 'interaction' ou utiliser pd.crosstab
clicks_pivot_binaire = pd.crosstab(
    index=clicks['user_id'], 
    columns=clicks['click_article_id']
)

# Convertir les valeurs > 1 en 1 (si besoin d'avoir strictement 0 ou 1)
clicks_pivot_binaire = (clicks_pivot_binaire > 0).astype(int)

display(clicks_pivot_binaire.head())


/var/folders/dm/pvk529ls4pg3bshgj9btlsc80000gn/T/ipykernel_14467/2451538373.py:2: PerformanceWarning: The following operation may generate 3023256640 cells in the resulting pandas object.
  clicks_pivot_binaire = pd.crosstab(


click_article_id,27,81,137,143,271,290,343,388,442,540,...,363957,363967,363976,363980,363984,363992,364001,364017,364022,364046
user_id,,,,,,,,,,,,,,,,,,,,,
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
# 1. Initialisation (identique)
model = implicit.als.AlternatingLeastSquares(factors=50, iterations=20, regularization=0.1, random_state=42)

# 2. Entraînement : ON UTILISE USER-ITEM (et non plus item-user)
print("Entraînement du modèle en cours...")
model.fit(sparse_user_item)
print("Entraînement terminé !")

# 3. Recommandation (identique)
# On convertit le vrai user_id en index catégoriel
user_idx = list(user_mapping.keys())[list(user_mapping.values()).index(user_id_original)]

recommended_items_idx, scores = model.recommend(
    userid=user_idx, 
    user_items=sparse_user_item[user_idx], 
    N=n_recommendations
)

# 4. Affichage
print(f"\nTop {n_recommendations} recommandations pour l'utilisateur {user_id_original} :")
for idx, score in zip(recommended_items_idx, scores):
    vrai_article_id = article_mapping[int(idx)]
    print(f"- Article ID: {vrai_article_id} | Score de pertinence: {score:.4f}")


Entraînement du modèle en cours...


NameError: name 'sparse_user_item' is not defined

In [12]:
import implicit
import scipy.sparse as sparse
import pandas as pd
import numpy as np

# Note : on suppose ici que votre dataframe "clicks" est déjà chargée en mémoire
# Exemple : clicks = pd.read_csv('click_hour_concatenate_100.csv')

# ---------------------------------------------------------
# 1. PRÉPARATION DES DONNÉES & MAPPING
# ---------------------------------------------------------
# Compter le nombre d'interactions par utilisateur et par article
df_implicit = clicks.groupby(['user_id', 'click_article_id']).size().reset_index(name='weight')

# Convertir les identifiants en index contigus (0, 1, 2...)
df_implicit['user_id_cat'] = df_implicit['user_id'].astype("category").cat.codes
df_implicit['article_id_cat'] = df_implicit['click_article_id'].astype("category").cat.codes

# Sauvegarder les dictionnaires pour retrouver les vrais IDs à la fin
user_mapping = dict(enumerate(df_implicit['user_id'].astype("category").cat.categories))
article_mapping = dict(enumerate(df_implicit['click_article_id'].astype("category").cat.categories))

# ---------------------------------------------------------
# 2. CRÉATION DE LA MATRICE
# ---------------------------------------------------------
# Création de la matrice User-Item (Utilisateurs en lignes, Articles en colonnes)
sparse_user_item = sparse.csr_matrix(
    (df_implicit['weight'].astype(float), (df_implicit['user_id_cat'], df_implicit['article_id_cat']))
)

# ---------------------------------------------------------
# 3. ENTRAÎNEMENT DU MODÈLE ALS
# ---------------------------------------------------------
model = implicit.als.AlternatingLeastSquares(factors=50, iterations=20, regularization=0.1, random_state=42)

print("Entraînement du modèle en cours...")
# On entraîne le modèle sur la matrice User-Item
model.fit(sparse_user_item)
print("Entraînement terminé !")

# ---------------------------------------------------------
# 4. PRÉDICTION ET AFFICHAGE
# ---------------------------------------------------------
# Choix de l'utilisateur pour la recommandation
user_id_original = clicks['user_id'].iloc[0] 
n_recommendations = 5

# Convertir le vrai ID en index interne pour le modèle
user_idx = list(user_mapping.keys())[list(user_mapping.values()).index(user_id_original)]

# Récupérer les recommandations
recommended_items_idx, scores = model.recommend(
    userid=user_idx, 
    user_items=sparse_user_item[user_idx], 
    N=n_recommendations
)

# Afficher les recommandations avec les vrais IDs d'articles
print(f"\nTop {n_recommendations} recommandations pour l'utilisateur {user_id_original} :")
for idx, score in zip(recommended_items_idx, scores):
    # On caste idx en int() pour éviter l'erreur KeyError np.int32
    vrai_article_id = article_mapping[int(idx)]
    print(f"- Article ID: {vrai_article_id} | Score de pertinence: {score:.4f}")


Entraînement du modèle en cours...


100%|██████████| 20/20 [00:23<00:00,  1.18s/it]

Entraînement terminé !

Top 5 recommandations pour l'utilisateur 0 :
- Article ID: 96663 | Score de pertinence: 0.0746
- Article ID: 119592 | Score de pertinence: 0.0687
- Article ID: 207122 | Score de pertinence: 0.0391
- Article ID: 160474 | Score de pertinence: 0.0242
- Article ID: 284410 | Score de pertinence: 0.0130


In [18]:
clicks['user_id'].is_unique

list_user_id_unique = list(set(clicks['user_id']))
list_user_id_unique

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,


In [17]:
len(list_user_id_unique)

176840

je crée une boucle for, pour les 10 premiers utilisateurs, donne la matrice 

In [26]:
matrice_recommandation_collaborative_filtering = {}
for user_id in list_user_id_unique:
    user_idx = list(user_mapping.keys())[list(user_mapping.values()).index(user_id)]

    recommended_items_idx, scores = model.recommend(
        userid=user_idx, 
        user_items=sparse_user_item[user_idx], 
        N=n_recommendations)

    matrice_recommandation_collaborative_filtering[user_id] = recommended_items_idx

df_matrice_recommandation_collaborative_filtering = pd.DataFrame.from_dict(matrice_recommandation_collaborative_filtering, orient="index")
df_matrice_recommandation_collaborative_filtering.to_csv("df_matrice_recommandation_collaborative_filtering.csv", index=True, index_label="user_id")
